# IoT 23 Dataset Feature Engineering
## Load Dataset and Libraries

In [1]:
import pandas as pd
import numpy as np

iot_23 = '/kaggle/input/datasets/engraqeel/iot23preprocesseddata/iot23_combined_new.csv'
df_iot = pd.read_csv(iot_23, index_col=0, low_memory=False)

In [2]:
df_iot.head(5)

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label
0,1.536227e+09,CeqqKl3hyLQmO8LK98,192.168.100.111,17576.0,78.1.220.212,8081.0,tcp,-,3e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
1,1.536227e+09,C2oHQWo1EFGH8D9x7,192.168.100.111,17576.0,152.84.7.111,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
2,1.536227e+09,CJLVjs4BByG04mczXc,192.168.100.111,17576.0,173.36.41.67,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
3,1.536227e+09,C0z4uS9AWHDH2s4S7,192.168.100.111,17576.0,87.13.21.104,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
4,1.536227e+09,CxbNVk3liFNUIlqSPi,192.168.100.111,17576.0,99.110.163.140,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan


## Remove uncecessary features

In [3]:
features_to_keep = ['ts', 'proto', 'orig_ip_bytes', 'resp_ip_bytes', 'label']
df_filtered = df_iot[features_to_keep].copy()
df_filtered = df_filtered.rename(columns={"ts": "timestamp", "orig_ip_bytes": "src_bytes", "resp_ip_bytes": "dest_bytes"})
df_filtered.head(10)

,timestamp,proto,src_bytes,dest_bytes,label
0,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
1,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
2,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
3,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
4,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
5,1.536227e+09,tcp,80.0,0.0,Okiru
6,1.536227e+09,tcp,80.0,0.0,Okiru
7,1.536227e+09,tcp,80.0,0.0,Okiru
8,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan
9,1.536227e+09,tcp,80.0,0.0,PartOfAHorizontalPortScan


## Create IAT (Inter Arrival Time) Feature

In [4]:
# Sort the entire filtered dataset chronologically
df_filtered = df_filtered.sort_values('timestamp')

df_filtered['iat'] = df_filtered['timestamp'].diff().fillna(0)
df_filtered.head(10)

,timestamp,proto,src_bytes,dest_bytes,label,iat
24,1.532101e+09,tcp,4100.0,0.0,Benign,0.000000
3,1.532101e+09,icmp,48.0,0.0,Benign,95.201964
4,1.532101e+09,udp,0.0,660.0,Benign,1.000190
8,1.532101e+09,udp,760.0,760.0,Benign,31.649943
7,1.532101e+09,udp,760.0,760.0,Benign,0.994438
14,1.532101e+09,udp,760.0,760.0,Benign,0.063458
0,1.532101e+09,udp,134.0,253.0,Benign,1.013922
10,1.532101e+09,udp,760.0,760.0,Benign,0.000748
5,1.532101e+09,udp,76.0,0.0,Benign,0.005747
1,1.532101e+09,udp,134.0,253.0,Benign,0.049474


## Drop timestamp as it is no longer required

In [5]:
df_filtered.drop(columns=['timestamp'], inplace=True)

## Print all labels

In [6]:
print(df_filtered['label'].unique())

['Benign' 'C&C-HeartBeat' 'Okiru' 'PartOfAHorizontalPortScan' 'C&C' 'DDoS'
 'C&C-Torii' 'Okiru-Attack' 'C&C-FileDownload' 'Attack' 'FileDownload'
 'C&C-HeartBeat-FileDownload' 'C&C-Mirai']


## Categorise labels into binary values for normal or attack

In [7]:
df_filtered['label'] = np.where(df_filtered['label'] == 'Benign', 0, 1)

In [8]:
print(df_filtered['label'].value_counts())

label
1    5357811
0     688812
Name: count, dtype: int64


## Downsample both labels so there are 25000 of each category

In [9]:
target_size = 25000 

# Separate the classes
df_normal = df_filtered[df_filtered['label'] == 0]
df_attack = df_filtered[df_filtered['label'] == 1]

# Downsample both to the target size
df_normal_light = df_normal.sample(n=target_size, random_state=42)
df_attack_light = df_attack.sample(n=target_size, random_state=42)

# Combine and Shuffle
df_ultra_light = pd.concat([df_normal_light, df_attack_light])
df_ultra_light = df_ultra_light.sample(frac=1, random_state=42).reset_index(drop=True)

# Verification
print(df_ultra_light['label'].value_counts())
print(f"Total rows: {len(df_ultra_light)}")

label
1    25000
0    25000
Name: count, dtype: int64
Total rows: 50000


## Convert protocol values into integer values. 1 - TCP, 2 - UDP, 3 - ICMP

In [10]:
df_ultra_light['proto'].unique()

array(['tcp', 'udp', 'icmp'], dtype=object)

In [11]:
# Define the mapping (TCP=6, UDP=17, ICMP=1, Others=0)
proto_mapping = {
    'tcp': 6,
    'udp': 17,
    'icmp': 1
}

# Apply the mapping
# .fillna(0) handles any protocols that aren't in the dictionary
df_ultra_light['proto'] = df_ultra_light['proto'].map(proto_mapping).fillna(0).astype(int)

# Check the result
print(df_ultra_light['proto'].value_counts())

proto
6     49322
17      613
1        65
Name: count, dtype: int64


In [12]:
df_ultra_light.dtypes

proto           int64
src_bytes     float64
dest_bytes    float64
label           int64
iat           float64
dtype: object

## Use logarithmic scaling for skewed columns

In [13]:
# Apply Log Scaling to the skewed columns
# Use log1p to safely handle the 0.0 values
df_ultra_light['iat_log'] = np.log1p(df_ultra_light['iat'])
df_ultra_light['src_bytes_log'] = np.log1p(df_ultra_light['src_bytes'])
df_ultra_light['dest_bytes_log'] = np.log1p(df_ultra_light['dest_bytes'])

# Drop the old unscaled columns to keep the file light
df_final = df_ultra_light.drop(columns=['iat', 'src_bytes', 'dest_bytes'])

# Check the new distribution
print(df_final.describe())

              proto         label       iat_log  src_bytes_log  dest_bytes_log
count  50000.000000  50000.000000  5.000000e+04   50000.000000    50000.000000
mean       6.128360      0.500000  3.233065e-02       4.067563        0.042738
std        1.224546      0.500005  3.035990e-01       1.118100        0.452602
min        1.000000      0.000000  7.152555e-07       0.000000        0.000000
25%        6.000000      0.000000  3.099437e-06       3.713572        0.000000
50%        6.000000      0.500000  5.006778e-06       4.110874        0.000000
75%        6.000000      1.000000  2.376751e-04       4.394449        0.000000
max       17.000000      1.000000  1.494617e+01      18.752700        8.776013


## Export dataframe to csv

In [14]:
df_final.to_csv('iot23_light_training.csv', index=False)